# 01 – Bronze: load the 3 raw CSVs as staging tables
Tab-separated, **no header** (see instawdb.sql BULK INSERT). Columns stay `_c0, _c1, ...` – they are named in Silver.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"                               # your catalog
RAW     = "/Volumes/workspace/default/raw"            # folder with the CSV files
SEP     = "\t"                                      # instawdb.sql: FIELDTERMINATOR = '\t'

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

display(dbutils.fs.ls(RAW))        # check the exact file names

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/raw/Product (1).csv,Product (1).csv,31122,1793468824000
dbfs:/Volumes/workspace/default/raw/Product (2).csv,Product (2).csv,88536,1793471809000
dbfs:/Volumes/workspace/default/raw/PurchaseOrderDetail.csv,PurchaseOrderDetail.csv,868775,1793471810000
dbfs:/Volumes/workspace/default/raw/PurchaseOrderHeader.csv,PurchaseOrderHeader.csv,516974,1793471810000
dbfs:/Volumes/workspace/default/raw/Region.csv,Region.csv,394,1793468824000
dbfs:/Volumes/workspace/default/raw/Reseller.csv,Reseller.csv,51958,1793468824000
dbfs:/Volumes/workspace/default/raw/Sales.csv,Sales.csv,4311826,1793468829000
dbfs:/Volumes/workspace/default/raw/Targets.csv,Targets.csv,36122,1793468824000
dbfs:/Volumes/workspace/default/raw/Vendor.csv,Vendor.csv,7161,1793471810000


In [0]:
files = {                                          # file name -> bronze table
    "PurchaseOrderDetail.csv": "stg_purchase_order_detail",
    "PurchaseOrderHeader.csv": "stg_purchase_order_header",
    "Vendor.csv":              "dim_vendor",
}

for f, table in files.items():
    df = (spark.read.option("header", False).option("sep", SEP)
          .option("encoding", "UTF-8").csv(f"{RAW}/{f}"))      # all strings: _c0, _c1, ...
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
      .saveAsTable(f"{CATALOG}.bronze.{table}")
    print(table, df.count(), "rows,", len(df.columns), "columns")
    display(df.limit(3))

stg_purchase_order_detail 8845 rows, 11 columns


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10
1,1,2022-04-29 00:00:00.000,4,1,50.2600,201.0400,3.00,.00,3.00,2022-04-22 00:00:00.000
2,2,2022-04-29 00:00:00.000,3,359,45.1200,135.3600,3.00,.00,3.00,2022-04-22 00:00:00.000
2,3,2022-04-29 00:00:00.000,3,360,45.5805,136.7415,3.00,.00,3.00,2022-04-22 00:00:00.000


stg_purchase_order_header 4012 rows, 13 columns


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12
1,5,4,258,1580,3,2022-04-15 00:00:00.000,2022-04-24 00:00:00.000,201.0400,16.0832,5.0260,222.1492,2022-04-24 00:00:00.000
2,5,1,254,1496,5,2022-04-15 00:00:00.000,2022-04-24 00:00:00.000,272.1015,21.7681,6.8025,300.6721,2022-04-24 00:00:00.000
3,5,4,257,1494,2,2022-04-15 00:00:00.000,2022-04-24 00:00:00.000,8847.3000,707.7840,221.1825,9776.2665,2022-04-24 00:00:00.000


dim_vendor 104 rows, 8 columns


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7
1492,AUSTRALI0001,Australia Bike Retailer,1,1,1,null,2022-12-22 00:00:00.000
1494,ALLENSON0001,Allenson Cycles,2,1,1,null,2022-04-24 00:00:00.000
1496,ADVANCED0001,Advanced Bicycles,1,1,1,null,2022-04-24 00:00:00.000
